# Machine Learning

Importación de librerías y modelos.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score


# Para guardar el modelo
import pickle

## Carga de datos

En el notebook anterior guardamos nuestros datos procesados en el directorio `data/`. Carguemos estos datos a un DataFrame.

> Nota: en este proyecto el archivo procesado se llama `titanic_feature.csv` (equivalente al `titanic_procesado.csv` del material del curso).

In [2]:
df = pd.read_csv('./data/titanic_feature.csv')
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Emb_C,Emb_Q,Emb_S
0,0,3,0,22.0,1,0,7.2500,False,False,True
1,1,1,1,38.0,1,0,71.2833,True,False,False
2,1,3,1,26.0,0,0,7.9250,False,False,True
3,1,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,0,35.0,0,0,8.0500,False,False,True


## División de datos

Dividimos nuestros datos en dos conjuntos: uno para **entrenamiento** y otro para **pruebas**. El de entrenamiento ajusta el modelo; el de pruebas evalúa que generalice bien a datos no vistos.

### train_test_split

Separamos las variables predictoras (`X`, sin la variable objetivo) de la variable objetivo (`y` = `Survived`).

In [3]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Emb_C,Emb_Q,Emb_S
0,3,0,22.0,1,0,7.2500,False,False,True
1,1,1,38.0,1,0,71.2833,True,False,False
2,3,1,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,3,0,35.0,0,0,8.0500,False,False,True


In [4]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

### Normalización con MinMaxScaler

Escalamos las variables al rango **[0, 1]** con `MinMaxScaler` (en el material los datos ya venían procesados/normalizados).

In [5]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Emb_C,Emb_Q,Emb_S
0,1.0,0.0,0.271174,0.125,0.0,0.014151,0.0,0.0,1.0
1,0.0,1.0,0.472229,0.125,0.0,0.139136,1.0,0.0,0.0
2,1.0,1.0,0.321438,0.000,0.0,0.015469,0.0,0.0,1.0
3,0.0,1.0,0.434531,0.125,0.0,0.103644,0.0,0.0,1.0
4,1.0,0.0,0.434531,0.000,0.0,0.015713,0.0,0.0,1.0


Hagamos la división entrenamiento-prueba:

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train.values  # Convertir a NumPy array
y_train = y_train.values  # Convertir a NumPy array
X_test = X_test.values    # Convertir a NumPy array
y_test = y_test.values    # Convertir a NumPy array

## Selección de modelo

Importamos una gran cantidad de modelos de Scikit-Learn. El código que implementaremos nos permitirá probarlos todos y elegir el mejor con base en las métricas de precisión, no en la intuición.

## GridSearch y Entrenamiento

Implementaremos **GridSearch**, que explora exhaustivamente combinaciones de hiperparámetros y selecciona la de mejor rendimiento. Primero creamos un diccionario con los modelos y los hiperparámetros a probar en cada uno.

In [7]:
# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'splitter': ['best', 'random'],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'n_neighbors': [3, 5, 7]
        }
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3],
            'learning_rate': [0.1, 0.2, 0.3],
            'verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'alpha': [0.1, 1.0, 10.0]
        }
    }
}

Ejecutamos el código completo que ajusta cada uno de los modelos con `GridSearchCV` (esto puede tardar ~1 minuto). No te preocupes si ves *warnings*: no interfieren con el entrenamiento.

In [8]:
# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    # Inicializar GridSearchCV con el modelo y los hiperparámetros
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'],
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)

    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)

    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)

    # Almacenar los resultados del modelo
    puntajes_modelos.append({'Modelo': nombre, 'Precisión': precision})
    estimadores[nombre] = grid_search.best_estimator_

    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/linear_model/_

/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/linear_model/_

/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarn

/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarn

/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarn

...

/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarn

/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/mhernandex/Downloads/aprendizaje_supervisado-main/2.2_analisis_exploratorio_de_datos/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Rendimiento de los modelos de clasificación
                                 Modelo  Precisión
6      Clasificador K-Nearest Neighbors       0.80
8                     Clasificador LGBM       0.80
1   Clasificador de Vectores de Soporte       0.80
2     Clasificador de Árbol de Decisión       0.79
4     Clasificador de Gradient Boosting       0.79
3    Clasificador de Bosques Aleatorios       0.79
5                 Clasificador AdaBoost       0.79
7                  Clasificador XGBoost       0.79
9                            GaussianNB       0.78
10             Clasificador Naive Bayes       0.77
0                   Regresión Logística       0.77
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.80


## Inferencia

Usamos el mejor estimador para predecir datos nuevos. El método `predict` recibe un numpy array con **las mismas dimensiones** que `X_train`. Tomamos el primer registro de `X_train` como ejemplo.

> Nota: nuestro `X` tiene 9 columnas (`Embarked` se expandió en `Emb_C/Emb_Q/Emb_S`), por eso construimos `nuevos_datos` a partir de `X_train[0]` real, en lugar de los 7 valores fijos del material.

In [9]:
X_train[0]

array([0.        , 0.        , 0.56647399, 0.        , 0.        ,
       0.0556283 , 0.        , 0.        , 1.        ])

In [10]:
y_train[0]

np.int64(0)

In [11]:
nuevos_datos = np.array(X_train[0]).reshape(1, -1)
nuevos_datos

array([[0.        , 0.        , 0.56647399, 0.        , 0.        ,
        0.0556283 , 0.        , 0.        , 1.        ]])

In [12]:
mejor_estimador.predict(nuevos_datos)

array([0])

## Guardar el modelo

Guardamos el mejor estimador con **Pickle** para usarlo en la siguiente lección (puesta en producción).

In [13]:
with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

print('Modelo guardado en modelo.pkl')

Modelo guardado en modelo.pkl
